This notebook builds on top of findings in the modeling_v1 notebook, refitting the binned months from preprocessing_v2 on a new logistic regression model.

In [19]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.linear_model import LogisticRegressionCV
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

In [20]:
df = pd.read_csv('../data/clean_online_shoppers_dataset_v2.csv')

X = df.drop(columns=['Revenue'])
y = df['Revenue']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size = 0.2,
    stratify = y,
    random_state = 42
)

print("Train shape:", X_train.shape, y_train.shape)
print("Test shape:", X_test.shape, y_test.shape)
print("\nClass distribution (train):")
print(y_train.value_counts(normalize=True))
print("\nClass distribution (test):")
print(y_test.value_counts(normalize=True))

Train shape: (9859, 27) (9859,)
Test shape: (2465, 27) (2465,)

Class distribution (train):
Revenue
False    0.845218
True     0.154782
Name: proportion, dtype: float64

Class distribution (test):
Revenue
False    0.84503
True     0.15497
Name: proportion, dtype: float64


In [21]:
binary_cols = [col for col in X_train.columns if X_train[col].nunique() == 2 and set(X_train[col].unique()).issubset({0,1})]

continuous_cols = [col for col in X_train.columns if col not in binary_cols]

print("Binary columns:", binary_cols)
print("Continuous columns:", continuous_cols)

Binary columns: ['TrafficType_2', 'TrafficType_3', 'TrafficType_4', 'TrafficType_5', 'TrafficType_6', 'TrafficType_8', 'TrafficType_10', 'TrafficType_11', 'TrafficType_13', 'TrafficType_20', 'TrafficType_Other', 'Month_Dec', 'Month_Mar', 'Month_May', 'Month_Nov', 'VisitorType_New_Visitor', 'is_special_day', 'Visited_Informational', 'has_bounce', 'high_exit_rate']
Continuous columns: ['Administrative', 'Informational', 'ProductRelated', 'ExitRates', 'Administrative_Duration_log', 'PageValues_log', 'ProductRelated_Duration_sqrt']


In [22]:
scaler = RobustScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[continuous_cols] = scaler.fit_transform(X_train[continuous_cols])
X_test_scaled[continuous_cols] = scaler.transform(X_test[continuous_cols])

print(X_train_scaled[continuous_cols].describe())
print("\nBinary cols untouched, sample:")
print(X_train_scaled[binary_cols].head())

       Administrative  Informational  ProductRelated    ExitRates  \
count     9859.000000    9859.000000     9859.000000  9859.000000   
mean         0.330510       0.505427        0.444228     0.500072   
std          0.833041       1.256691        1.409018     1.353127   
min         -0.250000       0.000000       -0.580645    -0.700000   
25%         -0.250000       0.000000       -0.354839    -0.300000   
50%          0.000000       0.000000        0.000000     0.000000   
75%          0.750000       0.000000        0.645161     0.700000   
max          6.500000      16.000000       18.258065     4.900000   

       Administrative_Duration_log  PageValues_log  \
count                  9859.000000     9859.000000   
mean                      0.047861        0.624058   
std                       0.520461        1.267545   
min                      -0.459295        0.000000   
25%                      -0.459295        0.000000   
50%                       0.000000        0.000000   


## Logistic Regression (With Cross-Validated Regularization)

In [23]:
log_reg_final = LogisticRegressionCV(
    l1_ratios=[1.0],
    solver='liblinear',
    Cs=10,
    cv=5,
    random_state=42,
    max_iter=1000,
    scoring='roc_auc', # goes beyond measuring 'c' scores by accuracy (Revenue in the dataset is imbalanced)
    class_weight='balanced'
)

log_reg_final.fit(X_train_scaled, y_train)

print("Best C:", log_reg_final.C_)

e:\Miniconda\envs\da\Lib\site-packages\sklearn\linear_model\_logistic.py:1823: FutureWarning: The fitted attributes of LogisticRegressionCV will be simplified in scikit-learn 1.10 to remove redundancy. Set`use_legacy_attributes=False` to enable the new behavior now, or set it to `True` to silence this warning during the transition period while keeping the deprecated behavior for the time being. The default value of use_legacy_attributes will change from True to False in scikit-learn 1.10. See the docstring of LogisticRegressionCV for more details.
  warnings.warn(


Best C: [2.7825594]


In [24]:
y_pred = log_reg_final.predict(X_test_scaled)
y_pred_proba = log_reg_final.predict_proba(X_test_scaled)[:, 1]

print("ROC AUC:", roc_auc_score(y_test, y_pred_proba))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

ROC AUC: 0.9228245105604332

Classification Report:
              precision    recall  f1-score   support

       False       0.96      0.88      0.92      2083
        True       0.55      0.80      0.66       382

    accuracy                           0.87      2465
   macro avg       0.76      0.84      0.79      2465
weighted avg       0.90      0.87      0.88      2465


Confusion Matrix:
[[1835  248]
 [  75  307]]


Out of all sessions that did end in a purchase, the model accurately flagged 80% of them.

In [25]:
coef_df = pd.DataFrame({
    'feature': X_train_scaled.columns,
    'coefficient': log_reg_final.coef_[0]
})
coef_df['abs_coef'] = coef_df['coefficient'].abs()
coef_df_sorted = coef_df.sort_values('abs_coef', ascending=False).reset_index(drop=True)

print(coef_df_sorted.to_string())

                         feature  coefficient  abs_coef
0                 PageValues_log     1.270135  1.270135
1                      Month_May    -1.158958  1.158958
2                 high_exit_rate    -0.984515  0.984515
3                      Month_Mar    -0.957106  0.957106
4                  TrafficType_5     0.806154  0.806154
5                  TrafficType_8     0.761314  0.761314
6                      Month_Nov     0.691225  0.691225
7                      Month_Dec    -0.672690  0.672690
8                 TrafficType_20     0.661370  0.661370
9                 TrafficType_13    -0.552232  0.552232
10                TrafficType_11     0.511887  0.511887
11       VisitorType_New_Visitor     0.410203  0.410203
12                 TrafficType_2     0.330007  0.330007
13                 TrafficType_4     0.250090  0.250090
14         Visited_Informational     0.244088  0.244088
15                TrafficType_10     0.239029  0.239029
16                     ExitRates    -0.232319  0

In [26]:
coef_df_sorted['odds_ratio'] = np.exp(coef_df_sorted['coefficient'])
print(coef_df_sorted[['feature', 'coefficient', 'odds_ratio']].head(10))

          feature  coefficient  odds_ratio
0  PageValues_log     1.270135    3.561332
1       Month_May    -1.158958    0.313813
2  high_exit_rate    -0.984515    0.373621
3       Month_Mar    -0.957106    0.384003
4   TrafficType_5     0.806154    2.239280
5   TrafficType_8     0.761314    2.141088
6       Month_Nov     0.691225    1.996159
7       Month_Dec    -0.672690    0.510334
8  TrafficType_20     0.661370    1.937445
9  TrafficType_13    -0.552232    0.575664


### Coefficient Check

After re-running the model by binning the month of February together with other low-session months, it no longer shows up as a coefficient because it is now part of the baseline that May and 3 other months are compared against. This updated model shows more plausible coefficients, with the months that do appear in the table each backed by 1000+ sessions.

May now shows up at the top as the 2nd biggest influence on the odds of a session leading to revenue. All else being equal, the month of May decreases the likelihood of the session ending in a purchase.

March, November and December all show up amongst the top 8 features that most influence the odds of a purchase, with March and December decreasing the odds, while November increases them.

PageValues is the single largest contributor to purchase predictions- with higher page values being associated with higher odds of a purchase.

A high exit rate (>= 0.18) has the 3rd biggest coefficient, reducing the odds of a session leading to a purchase.

Traffic Types 5 and 8 are among the top 6 features that improve purchase rates.

The type, duration and number of various page types visited had a much smaller affect on purchase likelihood than I expected- none showing up among the top 10 coefficients.


### Odds Ratio

All else being equal, for sessions originating from traffic types 5, 8 and 20, the odds of a session ending in a purchase are 2.24x, 2.14x, and 1.94x that of other traffic sources, respectively.

Sessions with a high exit rate (>= 0.18) have 0.37x the odds of ending in a purchase compared to sessions without one.

A session in November has nearly 2x the odds of a session ending in a purchase compared to other months.

The odds of a session leading to a purchase in May, March, and December are 0.31x, 0.38x, and 0.51x that of other months, suggesting seasonality in purchase rates.

In [27]:
all_features = coef_df_sorted
all_features.to_csv('../results/data/all_features.csv')

top_10_features = coef_df_sorted[['feature', 'coefficient', 'odds_ratio']].head(10)
top_10_features.to_csv('../results/data/top_10_features.csv')